In [ ]:
# from langchain_community.utilities import WikipediaAPIWrapper
# from langchain_community.tools import WikipediaQueryRun

In [ ]:
# api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
# wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


In [44]:
# ==========================================
# 1. IMPORT STATEMENTS
# ==========================================
import json
from typing import Type, Optional, Any
from pydantic import BaseModel, Field

# LangChain Core imports for creating custom tools
from langchain_core.tools import BaseTool
from langchain_core.callbacks import CallbackManagerForToolRun

# Libraries for the tools themselves
import wikipedia
import arxiv

# ==========================================
# 2. CUSTOM WIKIPEDIA TOOL
# ==========================================
class WikipediaInput(BaseModel):
    query: str = Field(description="The search query to look up on Wikipedia.")

class CustomWikipediaTool(BaseTool):
    name: str = "wikipedia"
    description: str = "Use this tool to search for general knowledge, historical facts, or information on any person, place, or thing."
    args_schema: Type[BaseModel] = WikipediaInput

    def _run(
        self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None, **kwargs: Any
    ) -> str:
        """Synchronously search Wikipedia."""
        try:
            # Set a proper User-Agent as required by Wikipedia's policy
            wikipedia.set_user_agent("MyLangChainApp/1.0 (https://myapp.com; myemail@example.com)")
            
            # Search for the query
            search_results = wikipedia.search(query, results=3)
            if not search_results:
                return f"No Wikipedia results found for '{query}'."
            
            # Get the page content for the top result
            page_title = search_results[0]
            page = wikipedia.page(page_title, auto_suggest=False)
            
            # Clean and truncate the content
            content = page.content.replace('\n', ' ').strip()
            if len(content) > 2000:
                content = content[:2000] + "...(truncated)"
                
            return f"### Wikipedia: {page.title}\n\n{content}\n\nSource: {page.url}"
            
        except wikipedia.DisambiguationError as e:
            return f"Query was ambiguous. Possible options: {e.options[:5]}"
        except wikipedia.PageError:
            return f"No Wikipedia page found for query: '{query}'"
        except Exception as e:
            return f"Wikipedia search failed: {str(e)}"

# ==========================================
# 3. CUSTOM ARXIV TOOL
# ==========================================
class ArxivInput(BaseModel):
    query: str = Field(description="The search query for academic papers on arXiv (e.g., 'attention is all you need', 'quantum machine learning').")

class CustomArxivTool(BaseTool):
    name: str = "arxiv"
    description: str = "Use this tool to search for academic papers, scientific research, and preprints on arXiv (categories: cs, math, physics, etc.)."
    args_schema: Type[BaseModel] = ArxivInput

    def _run(
        self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None, **kwargs: Any
    ) -> str:
        """Synchronously search Arxiv."""
        try:
            # Set up the arXiv client
            client = arxiv.Client(
                page_size=1,
                delay_seconds=3.0,
                num_retries=3
            )
            
            # Perform the search
            search = arxiv.Search(
                query=query,
                max_results=1,
                sort_by=arxiv.SortCriterion.Relevance
            )
            
            result_texts = []
            for paper in client.results(search):
                # Format the result
                title = paper.title.replace('\n', ' ')
                summary = paper.summary.replace('\n', ' ').strip()
                if len(summary) > 1000:
                    summary = summary[:1000] + "..."
                    
                paper_text = (
                    f"### Title: {title}\n"
                    f"**Authors:** {', '.join(a.name for a in paper.authors)}\n"
                    f"**Published:** {paper.published.date()}\n\n"
                    f"**Summary:** {summary}\n\n"
                    f"**PDF URL:** {paper.pdf_url}"
                )
                result_texts.append(paper_text)
                
            if not result_texts:
                return f"No arXiv results found for query: '{query}'"
                
            return "\n\n---\n\n".join(result_texts)
            
        except Exception as e:
            return f"Arxiv search failed: {str(e)}"

# ==========================================
# 4. INSTANTIATE AND USE THE TOOLS
# ==========================================

# Create instances of your custom tools
custom_wiki_tool = CustomWikipediaTool()
custom_arxiv_tool = CustomArxivTool()

# You can now use these tools directly or pass them to your agent
# as you would with the original ones.
# Example: Combine with other tools for your agent
# all_tools = [custom_wiki_tool, custom_arxiv_tool]

# --- Your agent creation code remains the same ---
# agent = create_react_agent(llm, all_tools, prompt)
# agent_executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True)
# response = agent_executor.invoke({"input": "who is elon musk?"})

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
loaded_docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(loaded_docs)


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
vectordb = FAISS.from_documents(documents, OpenAIEmbeddings())
retriver = vectordb.as_retriever()

In [6]:
retriver

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000226A62D6F90>, search_kwargs={})

In [7]:
# I have made tool for wikipedia and retrivers from web base loaders and vector stores.
# Now we need a retriver tool to retrive the data from the tools & then we will use the retrived data to answer the question.
# this is a customized tool I have made for retrivel and I made it a tool as agent dont know direct query retrieval but it have to choose a tool and then under the tool hood we have the actual retriver. that is the reason we keep the retiver inside a tool wrapper
from langchain_classic.tools.retriever import create_retriever_tool
retriver_tool = create_retriever_tool(retriever=retriver, name="langsmith_search", description="search for information about langsmith. For any information about langsmith, use this tool to search for it and get the answer. Always use this tool to search for information about langsmith. You must use this tool to search for information about langsmith. You should not answer any question about langsmith without using this tool to search for the information. Always use this tool to search for information about langsmith. You must use this tool to search for information about langsmith.")


In [8]:
retriver_tool.name

'langsmith_search'

In [ ]:
# # arxiv tool is used for fetching the paper onmline research papers. like we did for wikipedia it is an inbuilt tool in langchain
# from langchain_community.utilities import ArxivAPIWrapper
# from langchain_community.tools import ArxivQueryRun

In [ ]:
# api_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
# arxiv_tool = ArxivQueryRun(api_wrapper=api_wrapper)

In [ ]:
# from langchain_community.tools import DuckDuckGoSearchRun

# # Create the tool
# duckduckgo_tool = DuckDuckGoSearchRun()


In [46]:
# now combine all the tool in a list and then we will pass it to the agent
# tools = [wiki_tool, arxiv_tool, retriver_tool]
tools = [custom_wiki_tool, custom_arxiv_tool, retriver_tool]
# tools = [duckduckgo_tool, retriver_tool]

In [13]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Islam\\Documents\\llm-krish\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='search for information about langsmith. For any information about langsmith, use this tool to search for it and get the answer. Always use this tool to search for information about langsmith. You must use this tool to search for information about langsmith. You should not answer any question about langsmi

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model = "gpt-4.1-nano")  # nano


In [15]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("hwchase17/openai-functions-agent", dangerously_pull_public_prompt=True)

print(type(prompt))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [16]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [47]:
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm, tools, prompt)

In [48]:
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


In [51]:
agent_executor.invoke({"input": "what are the research papers quantum physics?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'quantum physics'}`


Arxiv search failed: Page request resulted in HTTP 301: The European Particle Physics Strategy Update (EPPSU) process takes a bottom-up approach, whereby the community is first invited to submit proposals (also called inputs) for projects that it would like to see realised in the near-term, mid-term and longer-term future. National inputs as well as inputs from National Laboratories are also an important element of the process. All these inputs are then reviewed by the Physics Preparatory Group (PPG), whose role is to organize a Symposium around the submitted ideas and to prepare a community discussion on the importance and merits of the various proposals. The results of these discussions are then concisely summarised in this Briefing Book, prepared by the Conveners, assisted by Scientific Secretaries, and with further contributions provided by the Contributors listed on the title page. Thi

{'input': 'what are the research papers quantum physics?',
 'output': 'The search for research papers on quantum physics on arXiv did not return specific results in this instance. Would you like me to attempt to retrieve papers with more specific keywords or topics within quantum physics?'}

In [52]:
agent_executor.invoke({"input": "what is langsmith?"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'langsmith'}`


pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.

LangSmith docs - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizat

{'input': 'what is langsmith?',
 'output': 'LangSmith is a framework-agnostic platform designed for building, debugging, and deploying AI agents and large language model (LLM) applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments in one integrated environment. The platform supports various frameworks and providers, enabling users to create and manage AI-powered applications efficiently. LangSmith offers features such as observability, evaluation, deployment, and platform setup, catering to the entire lifecycle of AI application development from local development to production. Additionally, it provides tools like Fleet for visual designing and deploying AI agents, a CLI for terminal management, and Studio for end-to-end application design and refinement.'}

In [53]:
agent_executor.invoke({"input": "who is muhammad islam kamran"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Muhammad Islam Kamran'}`


### Wikipedia: Kamran Mirza

Kamran Mirza (Persian: کامران میرزا; c. 1509 – 16/18 October 1557) was the second son of Babur, the founder of the Mughal Empire and the first Mughal Emperor. Kamran Mirza was born in Kabul to Babur's wife Gulrukh Begum. He was half-brother to Babur's eldest son Humayun, who would go on and inherit the Mughal throne, but he was full-brother to Babur's third son, Askari. A divan written in Persian and Chagatai is attributed to him.   == Early life == Kamran Mirza was born in around 1509 to Babur and Gulrukh Begum, a lady of the Begchik Mughals. Kamran spent his early childhood in Kabul, where Babur's household was based during the years before the conquest of India. Kamran received the traditional Timurid aristocratic education, which included theology, Persian literature, history, calligraphy, martial arts, and equestrian skills. Babur personally overs

{'input': 'who is muhammad islam kamran',
 'output': 'Muhammad Islam Kamran appears to be a name associated with historical or scholarly figures, but based on the search, it seems there may not be specific information widely available about this exact name. It could refer to a person with that name who has not gained significant prominence or might be a less documented individual. \n\nIf you have more context or specific details about Muhammad Islam Kamran, please provide them, and I can assist further in identifying or explaining who he might be.'}